# Deposit IVDRS & SUDORS to Prism **production**

Deposits the 86 records on the workbook's **Metadata for Deposit** tab (Action = Deposit) with their files,
into a new parent community. Keep `prism_deposit.py` in the same folder as this notebook.

| Cell | What it does | Sends anything? |
|---|---|---|
| 1 | Install packages | – |
| 2 | Settings ✏️ | – |
| 3 | Load the workbook and check every file | no (offline) |
| 4 | Dry run: build every Prism payload and save them to look at | no (offline) |
| 5 | Check your token | read-only |
| 6 | Check resource types, licenses, roles and language against Prism | read-only |
| 7 | **Create the parent community** | only if `CREATE_COMMUNITY = True` |
| 8 | Invite community members (Kathleen Kelly, owner) | only if `ADD_MEMBERS = True` |
| 9 | **Test batch:** drafts + files for 3 records | only if `SEND_TEST_DRAFTS = True` |
| 10 | Publish the test batch into the community | only if `PUBLISH_TEST = True` |
| 11 | All records: drafts + files | only if `SEND_ALL_DRAFTS = True` |
| 12 | Publish all records into the community | only if `PUBLISH_ALL = True` |
| 13 | Status report | read-only |

Every sending cell has its own switch, all `False` to start, so *Run All* never changes anything in Prism.
Every step is safe to re-run: a draft that exists is updated (never duplicated), files already uploaded are skipped,
and published records are skipped. Progress is kept in `output/production_registry.json`; every action is logged in
`output/production_deposit_log.csv`.

**What is and isn't sent**
- Reports and the presentation: Prism mints the DOI. Journal articles: the publisher's DOI is sent as an *external*
  DOI, so Prism does not mint a second one.
- Subjects are sent the way Prism stores them (the Subjects tab's last column): controlled terms by id, the rest as keywords.
- Authors' ORCIDs are sent; Scopus author IDs and the source column are **not** (Prism holds ORCIDs only).
- Funding is **never** sent (the Funding tab is for reference).
- Child communities don't exist yet: records go into the parent community only. Their child-community names are
  already in each record's keywords for Guillaume.

**PRODUCTION.** Records published here are permanent and get real DOIs. Every cell that changes Prism also needs `CONFIRM_PRODUCTION = "DEPOSIT TO PRODUCTION"` in cell 2. Progress is kept in `output/production_registry.json`, separate from staging. Subject ids were checked on staging: if production's subject lists differ, the test batch in cell 9 shows it as a draft warning.


## Cell 1: Install (once)

In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "requests", "openpyxl"], check=True)
print("Done.")

Done.


## Cell 2: Settings ✏️

Your token is read from `config.py` and never printed. It needs this line (a production token with deposit permission):

```python
PRISM_PRODUCTION_TOKEN = "..."
```

In [2]:
from pathlib import Path
import json
import importlib
import prism_deposit as pd_
pd_ = importlib.reload(pd_)          # picks up a replaced prism_deposit.py without restarting

WORKBOOK    = Path("2026_10-22 IVDRS_SUDORS_Reconciled_Deposit_List.xlsx")
FILES_DIR   = Path("files")                  # holds ivdrs_reports/, sudors_reports/, articles/
CONFIG_FILE = Path("config.py")
OUTPUT_DIR  = Path("output")

ENV         = "production"                    # keeps staging and production progress in separate files
PRISM_URL   = "https://prism.northwestern.edu"
TOKEN_VAR   = "PRISM_PRODUCTION_TOKEN"
VERIFY_SSL  = True                           # always verify on production
CONFIRM_PRODUCTION = "DEPOSIT TO PRODUCTION"                      # production only: type DEPOSIT TO PRODUCTION to allow changes

COMMUNITY_SLUG  = "ivdrs-sudors"             # becomes part of the community's web address
COMMUNITY_TITLE = "Illinois Violent Death and Overdose Reporting Systems (IVDRS & SUDORS)"
COMMUNITY_DESCRIPTION = ("Reports, data briefs, fact sheets, presentations and peer-reviewed publications from the "
                         "Illinois Violent Death Reporting System (IVDRS) and the Illinois State Unintentional Drug "
                         "Overdose Reporting System (SUDORS).")

TEST_IDS = ["IVSU-0125", "IVSU-0048", "IVSU-0045"]   # an Infogram report, an article, the 16-file county reports

RESOURCE_TYPE_IDS = {"report": "text-report",                     # Prism's ids (found on staging 2026-10-23)
                     "journal article": "article-journal_article",
                     "presentation": "learning-presentation"}
RESERVE_PRISM_DOIS     = True   # reserve a Prism DOI on report/presentation drafts (articles keep the publisher DOI)
INCLUDE_JOURNAL_FIELDS = False  # True = also send journal title/volume/issue/pages, if Prism has journal fields

env = pd_.load_config(CONFIG_FILE)
for p in (WORKBOOK, FILES_DIR, CONFIG_FILE):
    print(f"{'found  ' if p.exists() else 'MISSING'}  {p.resolve()}")
print(f"{TOKEN_VAR} configured: {bool(env.get(TOKEN_VAR))}")
OUTPUT_DIR.mkdir(exist_ok=True)
reg = pd_.Registry(OUTPUT_DIR / f"{ENV}_registry.json")
log = pd_.Log(OUTPUT_DIR / f"{ENV}_deposit_log.csv")

# helpers used by cells 9-12 (defined here so any of those cells can be run on its own)
def send_drafts(ids):
    pd_.require_confirmation(ENV, CONFIRM_PRODUCTION)
    for rid in ids:
        try:
            st = pd_.save_draft(api, rid, payloads[rid], reg, log, reserve_doi=RESERVE_PRISM_DOIS)
            if st.get("id") and not st.get("published"):
                pd_.upload_files(api, rid, files[rid], reg, log)
        except pd_.PrismError as e:
            log(rid, "error", "FAILED", str(e)[:400])

def publish_ids(ids):
    pd_.require_confirmation(ENV, CONFIRM_PRODUCTION)
    for rid in ids:
        try:
            pd_.publish(api, rid, reg, log, community_id=COMMUNITY_ID)
        except pd_.PrismError as e:
            log(rid, "publish", "FAILED", str(e)[:400])

found    C:\Users\keg827\Documents\Github_Repos\Galter_Prism_IVDRS_SUDORS_Post_Request\2026_10-22 IVDRS_SUDORS_Reconciled_Deposit_List.xlsx
found    C:\Users\keg827\Documents\Github_Repos\Galter_Prism_IVDRS_SUDORS_Post_Request\files
found    C:\Users\keg827\Documents\Github_Repos\Galter_Prism_IVDRS_SUDORS_Post_Request\config.py
PRISM_PRODUCTION_TOKEN configured: True


## Cell 3: Load the workbook and check the files (offline)

In [3]:
recs, members = pd_.read_workbook(WORKBOOK)
files = pd_.check_files(recs, FILES_DIR)
from collections import Counter
print(f"{len(recs)} records to deposit:", dict(Counter(r['row']['Resource type'] for r in recs.values())))
print(f"{sum(len(v) for v in files.values())} files;  community members listed: {len(members)}")
bad = [(rid, k, prob) for rid, lst in files.items() for k, _, prob in lst if prob]
no_files = [rid for rid, lst in files.items() if not lst]
for rid, k, prob in bad:
    print(f"  FILE PROBLEM  {rid}  {k}: {prob}")
for rid in no_files:
    print(f"  NO FILES      {rid}")
if not bad and not no_files:
    print("Every record has its file(s), and every file opens as a PDF.")

86 records to deposit: {'report': 46, 'presentation': 1, 'journal article': 39}
116 files;  community members listed: 1
Every record has its file(s), and every file opens as a PDF.


## Cell 4: Dry run (offline)

Builds every payload exactly as it will be sent and saves them to `output/production_payloads_dryrun.json`.
Vocabulary ids are assumed here; cell 6 checks them against Prism.

In [4]:
vocab = pd_.offline_vocab(recs, RESOURCE_TYPE_IDS)
payloads, problems = {}, {}
for rid, rec in recs.items():
    payloads[rid], problems[rid] = pd_.build_payload(rec, vocab, INCLUDE_JOURNAL_FIELDS)
(OUTPUT_DIR / "production_payloads_dryrun.json").write_text(json.dumps(payloads, indent=1, ensure_ascii=False), encoding="utf-8")
print("Saved", (OUTPUT_DIR / "production_payloads_dryrun.json").resolve())
for rid, p in problems.items():
    if p: print(f"  {rid}: " + "; ".join(p))
print("\nExample (article IVSU-0048):")
print(json.dumps(payloads["IVSU-0048"], indent=1, ensure_ascii=False)[:2500])

Saved C:\Users\keg827\Documents\Github_Repos\Galter_Prism_IVDRS_SUDORS_Post_Request\output\production_payloads_dryrun.json

Example (article IVSU-0048):
{
 "metadata": {
  "resource_type": {
   "id": "article-journal_article"
  },
  "title": "Firearm Restraining Order Implementation Case Study in Lake County, Illinois",
  "publication_date": "2025-09",
  "creators": [
   {
    "person_or_org": {
     "type": "personal",
     "family_name": "Mason",
     "given_name": "Maryann",
     "identifiers": [
      {
       "scheme": "orcid",
       "identifier": "0000-0001-6524-1532"
      }
     ]
    },
    "affiliations": [
     {
      "name": "Northwestern University, Chicago, IL, USA"
     }
    ]
   },
   {
    "person_or_org": {
     "type": "personal",
     "family_name": "Katz",
     "given_name": "Benjamin"
    },
    "affiliations": [
     {
      "name": "Northwestern University, Chicago, IL, USA"
     }
    ]
   },
   {
    "person_or_org": {
     "type": "personal",
     "family_

## Cell 5: Connect and check your token (read-only)

In [5]:
api = pd_.Prism(PRISM_URL, env.get(TOKEN_VAR, ""), verify=VERIFY_SSL)
try:
    mine = api.whoami()
    print(f"Token works. You have {mine.get('hits', {}).get('total', '?')} record(s)/draft(s) on production.")
except pd_.PrismError as e:
    print("Token check failed:", e)

Token works. You have 6419 record(s)/draft(s) on production.


## Cell 6: Check vocabularies against Prism (read-only)

Looks up each resource type, license, contributor role and language in Prism's own lists, then rebuilds the payloads
with the ids Prism actually uses. Anything not found is listed; licenses not in Prism's list go as custom licenses.

In [6]:
vocab = pd_.resolve_vocab(api, recs, RESOURCE_TYPE_IDS)
print("Resource types:", vocab["resource_types"])
print("Licenses:      ", vocab["licenses"])
print("Roles:         ", vocab["roles"])
print("Languages:     ", vocab["languages"])
for n in vocab["notes"]:
    print("  NOTE:", n)
payloads, problems = {}, {}
for rid, rec in recs.items():
    payloads[rid], problems[rid] = pd_.build_payload(rec, vocab, INCLUDE_JOURNAL_FIELDS)
(OUTPUT_DIR / "production_payloads.json").write_text(json.dumps(payloads, indent=1, ensure_ascii=False), encoding="utf-8")
blocked = [rid for rid, p in problems.items() if any("no Prism id" in x or "no creators" in x or "no publication date" in x for x in p)]
for rid, p in problems.items():
    if p: print(f"  {rid}: " + "; ".join(p))
print(f"\nReady: {len(recs) - len(blocked)} of {len(recs)}" + (f"   Blocked: {blocked}" if blocked else ""))

Resource types: {'journal article': 'article-journal_article', 'presentation': 'learning-presentation', 'report': 'text-report'}
Licenses:       {'cc-by-2.0': 'cc-by-2.0', 'cc-by-4.0': 'cc-by-4.0', 'cc-by-nc-4.0': 'cc-by-nc-4.0', 'cc-by-nc-nd-3.0': 'cc-by-nc-nd-3.0', 'cc-by-nc-nd-4.0': 'cc-by-nc-nd-4.0', 'cc-pdm-1.0': 'cc-pdm-1.0'}
Roles:          {'Other': 'role-other', 'role-hosting-institution': 'role-hosting-institution'}
Languages:      {'eng': True}

Ready: 86 of 86


## Cell 7: Create the parent community

Create the community on production (staging's community does not carry over). With `CREATE_COMMUNITY = False` this only checks; set it to `True` and
re-run to create it. You become its owner. If Prism says you aren't allowed to create communities, ask Guillaume to
create it with the slug above and make you an owner, then re-run this cell (it will find it).
The three child communities are left for Guillaume.

In [8]:
CREATE_COMMUNITY = True
if CREATE_COMMUNITY: pd_.require_confirmation(ENV, CONFIRM_PRODUCTION)
community = pd_.ensure_community(api, COMMUNITY_SLUG, COMMUNITY_TITLE, COMMUNITY_DESCRIPTION, send=CREATE_COMMUNITY)
COMMUNITY_ID = community["id"] if community else None

Created community: Illinois Violent Death and Overdose Reporting Systems (IVDRS & SUDORS)  (id 4e8d0998-aa45-4fde-a355-b435f48aca69, slug ivdrs-sudors). You are its owner.


## Cell 8: Invite community members

From the workbook's Community Members tab. A person can only be found after they have logged in to this Prism
(production) once. They get an invitation to accept.

In [9]:
ADD_MEMBERS = True
for m in members:
    print(f"{m['name']}  {m.get('netid')}  role: {m['role']}")
if COMMUNITY_ID:
    if ADD_MEMBERS: pd_.require_confirmation(ENV, CONFIRM_PRODUCTION)
    pd_.invite_members(api, COMMUNITY_ID, members, send=ADD_MEMBERS)
else:
    print("Create the community first (cell 7).")

Kathleen Kelly  kka1760  role: owner
NOT FOUND: Kathleen Kelly (kka1760) as owner. They need to log in to this Prism once; then re-run this cell. (Or invite them from the community's Members page.)


## Cell 9: Test batch: drafts and files (3 records)

Creates (or updates) a draft for each test record, reserves a Prism DOI for the report, and uploads its files.
Then look at the drafts in Prism (**My dashboard → Uploads**) before going on.

In [10]:
SEND_TEST_DRAFTS = True


if SEND_TEST_DRAFTS:
    send_drafts(TEST_IDS)
else:
    print("Not sent. Set SEND_TEST_DRAFTS = True to send:", TEST_IDS)

  IVSU-0125  draft          created  14fb8-6he32
  IVSU-0125  doi            reserved Prism DOI reserved
  IVSU-0125  file           uploaded IVSU-0125_Suicide_Deaths_Among_Females_in_Illinois_2018_2023.pdf (3,303,411 bytes)
  IVSU-0048  draft          created  fve2e-1vf96
  IVSU-0048  file           uploaded IVSU-0048_Firearm_Restraining_Order_Implementation_Case_Study_in_Lake.pdf (385,234 bytes)
  IVSU-0045  draft          created  e9z6z-x1216
  IVSU-0045  doi            reserved Prism DOI reserved
  IVSU-0045  file           uploaded IVSU-0045_Cook_County_2018.pdf (259,754 bytes)
  IVSU-0045  file           uploaded IVSU-0045_DuPage_County_2018.pdf (262,794 bytes)
  IVSU-0045  file           uploaded IVSU-0045_Effingham_County_2018.pdf (256,464 bytes)
  IVSU-0045  file           uploaded IVSU-0045_Kane_County_2018.pdf (260,790 bytes)
  IVSU-0045  file           uploaded IVSU-0045_Kankakee_County_2018.pdf (263,533 bytes)
  IVSU-0045  file           uploaded IVSU-0045_Kendall_County_2

## Cell 9b: Find out why Prism rejected a record (only if cell 9 shows FAILED)

Sends the record's payload with one part left out at a time, to see which part Prism refuses.
Test drafts that Prism accepts are deleted straight away; nothing is kept and progress isn't changed.
Send Claude the output.

In [11]:
DIAGNOSE_IDS = ["IVSU-0048"]
RUN_DIAGNOSIS = False
GOOD_ID = "IVSU-0125"          # a record Prism accepted in cell 9
if RUN_DIAGNOSIS:
    for rid in DIAGNOSE_IDS:
        pd_.diagnose_from_good(api, GOOD_ID, payloads[GOOD_ID], rid, payloads[rid])
else:
    print("Set RUN_DIAGNOSIS = True to test:", DIAGNOSE_IDS)

Set RUN_DIAGNOSIS = True to test: ['IVSU-0048']


## Cell 10: Publish the test batch into the community

Each draft is submitted to the community and the request is accepted (you're the owner), which publishes it there.
Publishing can't be undone (a published record can only get new versions), so check the drafts first.

In [12]:
PUBLISH_TEST = True


if PUBLISH_TEST:
    assert COMMUNITY_ID, "Create the community first (cell 7)."
    publish_ids(TEST_IDS)
else:
    print("Not published. Set PUBLISH_TEST = True to publish:", TEST_IDS)

  IVSU-0125  publish        done     10.18131/14fb8-6he32 https://prism.northwestern.edu/records/14fb8-6he32
  IVSU-0048  publish        done     10.1177/00469580251371370 https://prism.northwestern.edu/records/fve2e-1vf96
  IVSU-0045  publish        done     10.18131/e9z6z-x1216 https://prism.northwestern.edu/records/e9z6z-x1216


## Cell 11: All records: drafts and files

In [13]:
SEND_ALL_DRAFTS = True
if SEND_ALL_DRAFTS:
    send_drafts([rid for rid in recs if rid not in blocked])
else:
    print(f"Not sent. Set SEND_ALL_DRAFTS = True to send {len(recs) - len(blocked)} records.")

  IVSU-0125  draft          skip     already published as 14fb8-6he32 (edit/new version is a separate step)
  IVSU-0126  draft          created  cydw5-nnt20
  IVSU-0126  doi            reserved Prism DOI reserved
  IVSU-0126  file           uploaded IVSU-0126_Intimate_Partner_Problem_related_Violence_Deaths_in_Illinois.pdf (1,788,614 bytes)
  IVSU-0001  draft          created  ymmhk-4af41
  IVSU-0001  doi            reserved Prism DOI reserved
  IVSU-0001  file           uploaded IVSU-0001_Child_Abuse_Fatalities_in_Illinois_2015_2022.pdf (5,135,819 bytes)
  IVSU-0002  draft          created  7b6b1-k2d91
  IVSU-0002  doi            reserved Prism DOI reserved
  IVSU-0002  file           uploaded IVSU-0002_Veteran_Suicide_Deaths_in_Illinois_2019_2023.pdf (210,041 bytes)
  IVSU-0003  draft          created  j3xw5-k8x88
  IVSU-0003  doi            reserved Prism DOI reserved
  IVSU-0003  file           uploaded IVSU-0003_Intimate_Partner_Problem_Related_Suicide_and_Homicide_in_Ill.pdf (1,3

## Cell 12: Publish all records into the community

Only records whose draft exists and whose files are all uploaded are published. To hold some back, list them in `HOLD`.

In [15]:
PUBLISH_ALL = True
HOLD = []
if PUBLISH_ALL:
    assert COMMUNITY_ID, "Create the community first (cell 7)."
    publish_ids([rid for rid in recs if rid not in HOLD and rid not in blocked])
else:
    print("Not published. Set PUBLISH_ALL = True to publish.")

  IVSU-0125  publish        skip     already published: 10.18131/14fb8-6he32
  IVSU-0126  publish        done     10.18131/cydw5-nnt20 https://prism.northwestern.edu/records/cydw5-nnt20
  IVSU-0001  publish        done     10.18131/ymmhk-4af41 https://prism.northwestern.edu/records/ymmhk-4af41
  IVSU-0002  publish        done     10.18131/7b6b1-k2d91 https://prism.northwestern.edu/records/7b6b1-k2d91
  IVSU-0003  publish        done     10.18131/j3xw5-k8x88 https://prism.northwestern.edu/records/j3xw5-k8x88
  IVSU-0004  publish        done     10.18131/wg3nc-ba573 https://prism.northwestern.edu/records/wg3nc-ba573
  IVSU-0005  publish        done     10.18131/141mf-aje02 https://prism.northwestern.edu/records/141mf-aje02
  IVSU-0006  publish        done     10.18131/019zx-ysk49 https://prism.northwestern.edu/records/019zx-ysk49
  IVSU-0007  publish        done     10.18131/jgk78-nbx27 https://prism.northwestern.edu/records/jgk78-nbx27
  IVSU-0008  publish        done     10.18131/ds877

## Cell 13: Status report

In [16]:
import csv
rows = []
for rid in recs:
    st = reg.data.get(rid, {})
    rows.append({"record_id": rid, "title": recs[rid]["row"]["Title"], "draft_id": st.get("id"),
                 "files_uploaded": len(st.get("files", {})), "files_expected": len(files[rid]),
                 "draft_warnings": " | ".join(st.get("draft_errors", [])), "published": st.get("published", False),
                 "doi": st.get("doi"), "url": st.get("url")})
with open(OUTPUT_DIR / "production_status.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
c = Counter("published" if r["published"] else "draft" if r["draft_id"] else "not sent" for r in rows)
print(dict(c), "  ->", (OUTPUT_DIR / "production_status.csv").resolve())
for r in rows:
    if r["draft_warnings"]: print(f"  {r['record_id']}: {r['draft_warnings'][:300]}")

{'published': 86}   -> C:\Users\keg827\Documents\Github_Repos\Galter_Prism_IVDRS_SUDORS_Post_Request\output\production_status.csv


## After production

- **Child communities:** once Guillaume creates them, each record's child community (and the second one for IVDRS/SUDORS articles) is added as an inclusion request. That will be a short follow-up notebook.
- **Kathleen Kelly** needs to log in to production once before cell 8 can invite her.
